# 1 Introduction

- This script summarize R code used for modeling in the AI acceptance paper
- The input of data should be loaded separately (i.e.,preprocessed data in excel formats)
- If there's any question while running the scripts, please contact lincarrie32@gmail.com with details of the input data (format or example file), analysis goal and errors


#2 Environment Setup
- prepare packages we need

In [ ]:
# 1. Install/Load 'pacman' to manage libraries efficiently
if (!require("pacman")) install.packages("pacman")

# 2. Load core libraries (Colab comes with tidyverse pre-installed)
pacman::p_load(
  tidyverse,
  janitor,
  here,
  googledrive,  # Use this if you need to pull data from Drive
  readxl,
  writexl,
  lme4,
  lmerTest,
  emmeans,
  effectsize,
  ggplot2,
  ggeffects,
  dplyr,
  broom.mixed,
  ggpubr,
  rstatix
)

# 4. Print Session Info to verify versions for collaborators
sessionInfo()

Loading required package: pacman

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘snakecase’



janitor installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


here installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)


writexl installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rbibutils’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’



lme4 installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘numDeriv’



lmerTest installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘estimability’, ‘mvtnorm’



emmeans installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing t

R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] rstatix_0.7.3       ggpubr_0.6.3        broom.mixed_0.2.9.7
 [4] ggeffects_2.3.2     effectsize_1.0.1    emmeans_2.0.2      
 [7] lmerTest_3.2-1      lme4_2.0-1          Matr

#3 Modeling - linear mixed models & GLM
- part 1: for first parent group (v1p)
- part 2: for second parent group (v2p)
- part 3: cross-sample analysis (v1p_rep) using same predictors as part 1 but data from v2p
- part 4: for clinician group (v2c)
- part 5: group analysis between v2p & v2c

## part 1: first parent group
- the example file format is attached in 'example_data.xlsx'.
- it should be long-format, with Subject ID marked, and each column represents one predictor/variable,
each row represents one response per subject.

### DV1: general willingness

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "1_willingness_final.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "Response" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Gender_corr, response_type" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

In [ ]:
# 1. Load Libraries
if (!require("pacman")) install.packages("pacman")
pacman::p_load(performance)

# 2. Load and Clean
df <- read_excel(file_path)

# 3. Dynamic Mean-Centering of continuous predictors
# This looks for your specified variables and creates the '_c' versions
df <- df %>%
  mutate(across(all_of(center_list),
                ~ as.numeric(scale(., center = TRUE, scale = FALSE)),
                .names = "{.col}_c"))

# 4. Update Predictor List
# Automatically swaps original names for centered names if they exist in the DF
final_predictors <- map_chr(fixed_list, ~ {
  centered_name <- paste0(.x, "_c")
  if (centered_name %in% names(df)) centered_name else .x
})

# 5. Build and Run Formula
lmm_formula <- as.formula(paste(
  dependent_var, "~",
  paste(final_predictors, collapse = " + "),
  "+ (1 |", random_effect, ")"
))

model_full <- lmer(lmm_formula, data = df)

# 6. View Results
summary(model_full)

New names:
• `` -> `...1`


Linear mixed model fit by REML. t-tests use Satterthwaite's method [
lmerModLmerTest]
Formula: lmm_formula
   Data: df

REML criterion at convergence: 3741.3

Scaled residuals: 
    Min      1Q  Median      3Q     Max 
-3.5570 -0.5043  0.1779  0.6521  2.4008 

Random effects:
 Groups     Name        Variance Std.Dev.
 Subject_ID (Intercept) 2.132    1.460   
 Residual               5.031    2.243   
Number of obs: 792, groups:  Subject_ID, 198

Fixed effects:
                                Estimate Std. Error         df t value Pr(>|t|)
(Intercept)                     6.415207   0.164082 315.804056  39.098   <2e-16
Age_c                          -0.014755   0.022329 189.999998  -0.661   0.5095
Family_income_ordinal_c         0.027893   0.093505 189.999998   0.298   0.7658
Previous_AI_knowledge_score_c   0.245184   0.212649 189.999998   1.153   0.2504
Digital_usage_score_c           0.092220   0.094769 189.999998   0.973   0.3317
Years_of_education_c           -0.001004   0.055150 189.

In [ ]:
# save model output as csv files (with details for result reporting)

path_to_save = 'df1_v1p.xlsx' # specify your preferred path

results <- tidy(model_full)
write_xlsx(results, path_to_save)

### DV2: lowest acceptable accuracy - baseline accuracy

- can repeat the script above if specify the filepath of DV2
- other variables (fixed effects, centered var and random effects) stay the same with DV1's model

### DV3: maximal acceptable error rate
- since this variable is not repeatedly measured in our sample, so here we use Generalized Linear Model (GLM) with gamma log function for the model
- 'response_type' is removed here because there's only one DV3 for either type (no repeated measures)

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "3_error-rate_final.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "Response" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Gender_corr" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

In [ ]:
# 1. Load and Clean
df <- read_excel(file_path)

# 2. Dynamic Mean-Centering of continuous predictors
# This looks for your specified variables and creates the '_c' versions
df <- df %>%
  mutate(across(all_of(center_list),
                ~ as.numeric(scale(., center = TRUE, scale = FALSE)),
                .names = "{.col}_c"))

# 3. Update Predictor List
# Automatically swaps original names for centered names if they exist in the DF
final_predictors <- map_chr(fixed_list, ~ {
  centered_name <- paste0(.x, "_c")
  if (centered_name %in% names(df)) centered_name else .x
})

# 4. Build and Run Formula
glm_formula <- as.formula(paste(
  dependent_var, "~",
  paste(final_predictors, collapse = " + ")
))

model_glm <- glm(glm_formula, data = df, family= Gamma(link="log"))

# 5. View Results
summary(model_glm)

In [ ]:
# save model output as csv files (with details for result reporting)

path_to_save = 'df3_v1p.xlsx' # specify your preferred path

results <- tidy(model_glm)
write_xlsx(results, path_to_save)

### DV4: willingness at different data privacy levels
- new predictor for this DV: privacy levels
- the interaction effect between gender & privacy levels are also included

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "4_privacy_final.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Categorical Settings
categorical_var = "privacy_level" #@param {type:"string"}
#@markdown Which group should be the 'Baseline'? (e.g., privacy level 1)
reference_level = "1" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Gender_corr * privacy_level, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, response_type" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

In [ ]:
# Load the data
df <- read_excel(file_path)

# Convert Categorical Variable to Factor
# We use 'relevel' so that your results are compared to the baseline group you chose
df <- df %>%
  mutate(!!sym(categorical_var) := factor(!!sym(categorical_var)),
         !!sym(categorical_var) := relevel(!!sym(categorical_var), ref = reference_level))

df <- df %>%
  mutate(across(all_of(center_list),
                ~ as.numeric(scale(., center = TRUE, scale = FALSE)),
                .names = "{.col}_c"))

# Update Predictor List
# Automatically swaps original names for centered names if they exist in the DF
final_predictors <- map_chr(fixed_list, ~ {
  centered_name <- paste0(.x, "_c")
  if (centered_name %in% names(df)) centered_name else .x
})

# Build and Run Formula
lmm_formula <- as.formula(paste(
  dependent_var, "~",
  paste(final_predictors, collapse = " + "),
  "+ (1 |", random_effect, ")"
))

model_full <- lmer(lmm_formula, data = df)

# View Results
summary(model_full)

In [ ]:
# save model output as csv files (with details for result reporting)

path_to_save = 'df4_v1p.xlsx' # specify your preferred path

results <- tidy(model_full)
write_xlsx(results, path_to_save)

## part 2: second parent group
- in this participant group, three new predictors are added: perceived_severity, Previous_medical_knowledge, experience (yes/no)
- other variables are the same as in v1p

### DV1: general willingness

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "1_willingness_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Family_income_ordinal, Previous_medical_knowledge, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Gender_corr, response_type, experience, perceived_severity" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, perceived_severity, Previous_medical_knowledge" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

In [ ]:
# 2. Load and Clean
df <- read_excel(file_path)

# 3. Dynamic Mean-Centering of continuous predictors
# This looks for your specified variables and creates the '_c' versions
df <- df %>%
  mutate(across(all_of(center_list),
                ~ as.numeric(scale(., center = TRUE, scale = FALSE)),
                .names = "{.col}_c"))

# 4. Update Predictor List
# Automatically swaps original names for centered names if they exist in the DF
final_predictors <- map_chr(fixed_list, ~ {
  centered_name <- paste0(.x, "_c")
  if (centered_name %in% names(df)) centered_name else .x
})

# 5. Build and Run Formula
lmm_formula <- as.formula(paste(
  dependent_var, "~",
  paste(final_predictors, collapse = " + "),
  "+ (1 |", random_effect, ")"
))

model_full <- lmer(lmm_formula, data = df)

# 6. View Results
summary(model_full)

In [ ]:
# save model output as csv files (with details for result reporting)

path_to_save = 'df1_v2p.xlsx' # specify your preferred path

results <- tidy(model_full)
write_xlsx(results, path_to_save)

### DV2: lowest acceptable accuracy - baseline accuracy
- can repeat the script above if specify the filepath of DV2
- other variables (fixed effects, centered var and random effects) stay the same with DV1's model

### DV3: maximal acceptable error rate
- in this participant group, lmm is used for DV3 since there are repeated measures per subject
- 'response_type' is removed here because there's only one DV3 for either type (no repeated measures)

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "3_error_rate_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Family_income_ordinal, Previous_medical_knowledge, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Gender_corr, experience" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, perceived_severity, Previous_medical_knowledge" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

### DV4: willingness at different data privacy levels
- new predictor for this DV: privacy levels (categorical variable)
- the interaction effect between gender & privacy levels are also included

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "4_privacy_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Categorical Settings
categorical_var = "privacy_level" #@param {type:"string"}
#@markdown Which group should be the 'Baseline'? (e.g., privacy level 1)
reference_level = "1" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Gender_corr * privacy_level, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, response_type, Previous_medical_knowledge, experience, perceived_severity" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Previous_medical_knowledge, perceived_severity" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

## part 3: cross-sample analysis (v1p & v2p)
- in this analysis, we use the same set of predictors as in v1p and model them using data of second parent group (to check if there are replicated results)
- the formula stayed the same as in v1p, only the file_path is replaced with data from v2p
- for the modeling, just re-run part 1 using the following predictors for each DV

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "1_willingness_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, Gender_corr, response_type" #@param {type:"string"}
random_effect = "Subject_ID" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Family_income_ordinal, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

## part 4: clinician group
- for clinical group, the predictors are: working_experience (in years), age, Gender_corr, perceived_severity, Previous_AI_knowledge_score, Digital_usage_score, response_type
- the models are same as in v2p but with following predictors

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "1_willingness_v3_new.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Gender_corr, Previous_AI_knowledge_score, Digital_usage_score, Working_experience, perceived_severity, response_type" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Previous_AI_knowledge_score, Digital_usage_score, Working_experience, perceived_severity" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

## part 5: group analysis (v2p & v2c)
- in this analysis, we combined the data of second parent group and clinician group as one dataset, labeled with a new variable 'group' for each dataset to check if there are group differences across measured variables
- the shared predictors in v2p and v2c are used
- the interaction effect between digital usage & group is included (since both of their main effects are sig., it's reasonable to explore whether an interaction effect exist). The same rules apply to your predictors.
- repeat the models for DV1-DV4 (combined data) as in part 2 (v2p)

In [ ]:
#@title model configuration interface { display-mode: "form" }

#@markdown ### 📁 Data Source
file_path = "1_combined_data_willingness_v5.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variables of Interest
dependent_var = "response_score" #@param {type:"string"}
#@markdown Enter fixed effects separated by commas (e.g., Age, Gender)
fixed_effects_input = "Age, Gender_corr, Previous_AI_knowledge_score, perceived_severity, response_type, Digital_usage_score * group" #@param {type:"string"}

#@markdown ### 📏 Scaling Settings
#@markdown Which variables should be mean-centered? [hint: continuous variables]
vars_to_center = "Age, Previous_AI_knowledge_score, Digital_usage_score, perceived_severity" #@param {type:"string"}

# --- Process Inputs ---
fixed_list <- trimws(unlist(strsplit(fixed_effects_input, ",")))
center_list <- trimws(unlist(strsplit(vars_to_center, ",")))

#4 Environment Setup 2
for running brms models

In [2]:
if (!require("pacman")) install.packages("pacman")

pacman::p_load(
  brms,
  broom.mixed
)

# 4. Print Session Info to verify versions for collaborators
sessionInfo()

Loading required package: pacman

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘pacman’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘numDeriv’, ‘plyr’, ‘StanHeaders’, ‘inline’, ‘gridExtra’, ‘RcppParallel’, ‘QuickJSR’, ‘RcppEigen’, ‘BH’, ‘checkmate’, ‘tensorA’, ‘distributional’, ‘ggridges’, ‘reshape2’, ‘mvtnorm’, ‘Brobdingnag’, ‘globals’, ‘listenv’, ‘parallelly’, ‘rstan’, ‘loo’, ‘posterior’, ‘rstantools’, ‘bayesplot’, ‘bridgesampling’, ‘future’, ‘future.apply’, ‘matrixStats’, ‘nleqslv’, ‘coda’, ‘abind’



brms installed

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘furrr’



broom.mixed installed



R version 4.5.1 (2025-06-13)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 22.04.4 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.20.so;  LAPACK version 3.10.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
[1] broom.mixed_0.2.9.7 brms_2.23.0         Rcpp_1.1.0         

loaded via a namespace (and not attached):
 [1] bridgesampling_1.2-1  tensorA_0.36.2.1      future_1.69.0        


#5 Modeling - bayesian regression models

run for DV5 (decision made during AI-human inconsistency)
- part 1: for first parent group (v1p)
- part 2: for second parent group (v2p)
- part 3: cross-sample analysis (v1p_rep) using same predictors as part 1 but data from v2p
- part 4: for clinician group (v2c)
- part 5: group analysis between v2p & v2c

### part 1: first parent group

In [ ]:
#@title Bayesian Model Configuration { display-mode: "form" }

#@markdown ### 📁 Data Input
file_path = "5_decision_final.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variable Mapping
dependent_var = "Response" #@param {type:"string"}
categorical_var = "Gender_corr" #@param {type:"string"}
dv_reference_group = "B" #@param {type:"string"}

#@markdown ### 🧪 Model Structure
fixed_effects = "Age, Gender_corr, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Max_child_age, Years_of_education, response_type" #@param {type:"string"}
continuous_to_center = "Age, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}
random_intercept = "Subject_ID" #@param {type:"string"}

#@markdown ### ⚙️ Bayesian Settings
#@markdown Higher iterations = more accuracy but slower run time.
iterations = 5000 #@param {type:"slider", min:1000, max:5000, step:500}
chains = 4 #@param {type:"integer"}
adapt_delta_input = 0.95 #@param {type:"slider", min:0.8, max:0.99, step:0.01}

In [ ]:
# 1. Prepare Data

df <- read_excel(file_path)

df_clean <- df %>%
  # --- NEW: Relevel Dependent Variable ---
  mutate(!!sym(dependent_var) := factor(!!sym(dependent_var)),
         !!sym(dependent_var) := relevel(!!sym(dependent_var), ref = dv_reference_group)) %>%

  # --- Relevel Categorical Predictor ---
  mutate(!!sym(categorical_var) := factor(!!sym(categorical_var)),
         !!sym(categorical_var) := relevel(!!sym(categorical_var), ref = predictor_reference_group)) %>%

  # --- Mean Centering ---
  mutate(across(all_of(center_list),
                ~ as.numeric(scale(., center = TRUE, scale = FALSE)),
                .names = "{.col}_c"))

# 2. Formula Construction
final_predictors <- map_chr(fixed_list, function(term) {
  updated_term <- term
  for (var in center_list) {
    updated_term <- gsub(paste0("\\b", var, "\\b"), paste0(var, "_c"), updated_term)
  }
  return(updated_term)
})

bf_formula <- bf(paste(dependent_var, "~", paste(final_predictors, collapse = " + "),
                       "+ (1 |", random_intercept, ")"))

# 3. Run Bayesian Categorical Model
# Note: Using family = categorical() for 3+ factor levels
fit_brms <- brm(
  formula = bf_formula,
  data = df_clean,
  family = categorical(),
  chains = chains,
  iter = iterations,
  cores = parallel::detectCores(),
  control = list(adapt_delta = adapt_delta_input)
)

summary(fit_brms)

In [ ]:
# save the significant effects as csv file

# Tidy fixed effects
coefs <- broom.mixed::tidy(fit_brms, effects = "fixed")

# Mark significance if 95% CI excludes 0
coefs$sig <- ifelse(coefs$conf.low * coefs$conf.high > 0, TRUE, FALSE)

# Filter only significant
sig_coefs <- subset(coefs, sig == TRUE)

# Save -- ⚠️ change filename if needed
write.csv(sig_coefs, "significant_fixed_effects.csv", row.names = FALSE)

# check for 99.9% CI
# 99.9% CI for all fixed effects
posterior_summary(fit_brms, probs = c(0.0005, 0.9995))

### part 2: second parent group
- three new predictors for v2p: perceived_severity, Previous_medical_knowledge, experience
- to run the model: repeat the scripts in part 1 with updated file path and predictors

In [ ]:
#@title Bayesian Model Configuration { display-mode: "form" }

#@markdown ### 📁 Data Input
file_path = "5_decision_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variable Mapping
dependent_var = "response_score" #@param {type:"string"}
categorical_var = "Gender_corr, experience" #@param {type:"string"}
dv_reference_group = "B" #@param {type:"string"}

#@markdown ### 🧪 Model Structure
fixed_effects = "Age, Gender_corr, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Max_child_age, Years_of_education, response_type, perceived_severity, Previous_medical_knowledge " #@param {type:"string"}
continuous_to_center = "Age, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age, perceived_severity, Previous_medical_knowledge " #@param {type:"string"}
random_intercept = "Subject_ID" #@param {type:"string"}

#@markdown ### ⚙️ Bayesian Settings
#@markdown Higher iterations = more accuracy but slower run time.
iterations = 5000 #@param {type:"slider", min:1000, max:5000, step:500}
chains = 4 #@param {type:"integer"}
adapt_delta_input = 0.95 #@param {type:"slider", min:0.8, max:0.99, step:0.01}

### part 3: clinician group
- for clinical group, the predictors are: Working_experience (in years), age, Gender_corr, perceived_severity, Previous_AI_knowledge_score, Digital_usage_score, response_type

In [ ]:
#@title Bayesian Model Configuration { display-mode: "form" }

#@markdown ### 📁 Data Input
file_path = "5_decision_v3_new.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variable Mapping
dependent_var = "response_score" #@param {type:"string"}
categorical_var = "Gender_corr" #@param {type:"string"}
dv_reference_group = "B" #@param {type:"string"}

#@markdown ### 🧪 Model Structure
fixed_effects = "Age, Gender_corr, Previous_AI_knowledge_score, Digital_usage_score, perceived_severity, Working_experience " #@param {type:"string"}
continuous_to_center = "Age, Previous_AI_knowledge_score, Digital_usage_score, Working_experience,  " #@param {type:"string"}
random_intercept = "Subject_ID" #@param {type:"string"}

#@markdown ### ⚙️ Bayesian Settings
#@markdown Higher iterations = more accuracy but slower run time.
iterations = 5000 #@param {type:"slider", min:1000, max:5000, step:500}
chains = 4 #@param {type:"integer"}
adapt_delta_input = 0.95 #@param {type:"slider", min:0.8, max:0.99, step:0.01}

### part 4: cross-sample analysis of two parents group
- in this analysis, we use the same set of predictors as in v1p and model them using data of second parent group
- the predictors stayed the same as in v1p, only the file_path is replaced with data from v2p
- for the modeling, just re-run part 1 using the following predictors

In [ ]:
#@title Bayesian Model Configuration { display-mode: "form" }

#@markdown ### 📁 Data Input
file_path = "5_decision_v2.4.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variable Mapping
dependent_var = "response_score" #@param {type:"string"}
categorical_var = "Gender_corr" #@param {type:"string"}
dv_reference_group = "B" #@param {type:"string"}

#@markdown ### 🧪 Model Structure
fixed_effects = "Age, Gender_corr, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Max_child_age, Years_of_education, response_type" #@param {type:"string"}
continuous_to_center = "Age, Family_income_level, Previous_AI_knowledge_score, Digital_usage_score, Years_of_education, Max_child_age" #@param {type:"string"}
random_intercept = "Subject_ID" #@param {type:"string"}

#@markdown ### ⚙️ Bayesian Settings
#@markdown Higher iterations = more accuracy but slower run time.
iterations = 5000 #@param {type:"slider", min:1000, max:5000, step:500}
chains = 4 #@param {type:"integer"}
adapt_delta_input = 0.95 #@param {type:"slider", min:0.8, max:0.99, step:0.01}

### part 5: group analysis (v2p & v2c)
- in this analysis, we combined the data of second parent group and clinician group as one dataset, labeled with a new variable 'group' for each dataset to check if there are group differences across their decisions
- the shared predictors in v2p and v2c are used
- repeat the scripts in part 1 with updated file path and predictors to run the brm model

In [ ]:
#@title Bayesian Model Configuration { display-mode: "form" }

#@markdown ### 📁 Data Input
file_path = "5_combined_data_decision_v5.xlsx" #@param {type:"string"}

#@markdown ### 🎯 Variable Mapping
dependent_var = "response_score" #@param {type:"string"}
categorical_var = "Gender_corr, group" #@param {type:"string"}
dv_reference_group = "B" #@param {type:"string"}

#@markdown ### 🧪 Model Structure
fixed_effects = "Age, Gender_corr, Previous_AI_knowledge_score, Digital_usage_score, perceived_severity" #@param {type:"string"}
continuous_to_center = "Age, Previous_AI_knowledge_score, Digital_usage_score  " #@param {type:"string"}
random_intercept = "Subject_ID" #@param {type:"string"}

#@markdown ### ⚙️ Bayesian Settings
#@markdown Higher iterations = more accuracy but slower run time.
iterations = 5000 #@param {type:"slider", min:1000, max:5000, step:500}
chains = 4 #@param {type:"integer"}
adapt_delta_input = 0.95 #@param {type:"slider", min:0.8, max:0.99, step:0.01}